<a href="https://colab.research.google.com/github/tolumeadows/ai-portrait-pipeline/blob/main/lora_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive # imports drive from colab so we can utalize the drive folder with the paintings "LoRA"
import json # imports json so we can use the file data structure

drive.mount('/content/drive') # allows us to access our google drive
folder = ('/content/drive/MyDrive/LoRA_paintings') # saves the file path to our file of the painting images as name "folder" so we can use it
captions = """
Alligators.jpg: an oil painting in mystyle, alligators in the ocean at night
Birthday_Mom.jpg: an oil painting in mystyle, woman and child at a birthday party
Black_History.jpg: an oil painting in mystyle, faces of a boy and civil rights activists
Clown_Moth.jpg: an oil painting in mystyle, clown bending over in a field of grass with moth on picture
Empty_Room.jpg: an oil painting in mystyle, empty bed room with bed, curtain, table, chair
Face_Earth.jpg: an oil painting in mystyle, face of a boy on a black background
Kids_Wedding.jpg: an oil painting in mystyle, kids at a wedding after-party dressed in wedding attire
Lizards_Man.jpg: an oil painting in mystyle, people with lizard heads and a bleeding man sitting at a table
Me_Sleeping.jpg: an oil painting in mystyle, boy sleeping in a bed with white sheets and blanket
Pastor_Woman.jpg: an oil painting in mystyle, pastor laying his hand on a woman
Personal_Demons.jpg: an oil painting in mystyle, boy with avant-garde hat surrounded by people and objects
Secret_Woman.jpg: an oil painting in mystyle, woman standing at a vanity in her room
Self_Reflection.jpg: an oil painting in mystyle, Boy in front of a mirror looking at his reflection
Trinity.jpg: an oil painting in mystyle, a man and girl both in white drapes and a bronze statue of an angel hovering over them
Two_Men.jpg: an oil painting in mystyle, two men at a table looking across the room
"""

with open(f"{folder}/metadata.jsonl", "w") as f: # open json file in the LoRA_paintings folder, that we can write in and nicknames it f
  for line in captions.strip().splitlines(): # for each line in the captions string above, split each line:
    title, text = line.split(":", 1) # separate the title and text by the first ("1") colon
    f.write(json.dumps({"file_name": title.strip(), "text": text.strip()}) + "\n") # this writes into the json file this dictoriary of the name and text description of each piece. json.dumps converts the dict into a single line of json format text

print(open(f"{folder}/metadata.jsonl").read()) # opens the json file we just made

In [ ]:
# shows the GPU; should say Tesla T4
!nvidia-smi
# downloads the diffusers source code, including the training script
!git clone https://github.com/huggingface/diffusers
# moves into that folder
%cd diffusers
# installs diffusers from that source
!pip install .
# moves to the folder with the training script
%cd examples/text_to_image
# installs the extra libraries the script needs
!pip install -r requirements.txt
# sets up accelerate with default settings, no questions
!accelerate config default

In [ ]:
from google.colab import drive # connect drive again

drive.mount('/content/drive') # mount drive

!ls /content/drive/MyDrive/LoRA_paintings
!ls /content/drive/MyDrive/LoRA_paintings |  wc -l

In [ ]:
!pip uninstall -y torchao

In [ ]:
%cd /content/diffusers/examples/text_to_image

!accelerate launch --mixed_precision="no" train_text_to_image_lora.py \
--pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
--train_data_dir="/content/drive/MyDrive/LoRA_paintings" \
--caption_column="text" \
--resolution=512 \
--train_batch_size=1 \
--gradient_accumulation_steps=4 \
--gradient_checkpointing \
--max_train_steps=100 \
--learning_rate=1e-4 \
--output_dir="/content/drive/MyDrive/lora_test_output"